House Prices baseline (Ames, Iowa)

Blank-file rep on the Kaggle House Prices dataset (train.csv, 1460 rows, 79 features). Goal was a clean sklearn pipeline on messy data, with no leakage.

What I did

Dropped Id and SalePrice from X, took log1p of SalePrice as the target
Split first, then built everything inside one pipeline so imputers only learn from training data
Sorted columns into three groups:
none_cols: categoricals where NaN means "doesn't have one" (pool, garage, basement, alley, fence...), filled with the constant 'None'
cat_cols: the remaining categoricals, filled with most_frequent
num_cols: numeric columns, median imputer then scaler
OneHotEncoder with handle_unknown='ignore' (rare categories show up in CV folds, hence the warnings)
5-fold CV on the training set, then one final score on the held-out test set

Results (R2, log scale)

Model	CV mean	CV std	Test
Linear regression	0.783	0.054	0.859
Random forest	0.859	0.030	0.883

Taking the log of the target moved linear from 0.696 to 0.783 and cut its spread roughly in half. The forest was already coping with the skew, so it gained less (0.84 to 0.859).

Next: Ridge with a few alphas, min_frequency on the encoder, then feature engineering.

In [1]:
import pandas as pd 

path_train = '/home/arkalibaig/load_jl/train.csv'
df = pd.read_csv(path_train)

df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [3]:
df.isna().sum().sort_values(ascending=False).head(10)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
dtype: int64

In [4]:
df = df.drop(columns=['Id'])

In [5]:
from sklearn.model_selection import train_test_split, cross_val_score

X = df.drop(columns= ['SalePrice'])
y = np.log1p(df['SalePrice'])

X_train,X_test,y_train,y_test = train_test_split(X,y , test_size=0.2 , random_state =42)

NameError: name 'np' is not defined

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

num_cols = X_train.select_dtypes(include='number').columns.tolist()
none_cols = ['Fence', 'PoolQC', 'GarageCond', 'GarageQual', 'GarageType', 'GarageFinish',
             'FireplaceQu', 'BsmtFinType2', 'BsmtFinType1', 'BsmtExposure', 'BsmtCond',
             'BsmtQual', 'Alley', 'MiscFeature']
cat_cols = [c for c in cat_cols if c not in none_cols]

def make_prep():
    num_pipe = Pipeline([
        ('imputer' , SimpleImputer( strategy = 'median')),
        ('scaler' , StandardScaler())
    ])

    cat_pipe = Pipeline([
    ('imputer' , SimpleImputer( strategy= 'most_frequent')),
    ('encoder' , OneHotEncoder(drop='first', handle_unknown='ignore'))
   ])

    none_pipe = Pipeline([
        ('imputer' , SimpleImputer( strategy='constant' , fill_value='None')),
        ('encoder', OneHotEncoder(drop='first' , handle_unknown = 'ignore'))
        ])

    return ColumnTransformer([
        ('num', num_pipe , num_cols),
        ('cat' , cat_pipe, cat_cols),
        ('None' , none_pipe, none_cols)
    ])    

   
pipe_lr = Pipeline([
    ('prep' , make_prep()),
    ('model', LinearRegression())
])

pipe_rf = Pipeline([
    ('prep', make_prep()),
    ('model', RandomForestRegressor(random_state=42, max_depth =10))
])

for name, pipe in [('linear', pipe_lr), ('random', pipe_rf)]:
    scores = cross_val_score(pipe, X_train, y_train , cv=5, scoring='r2')
    print(name, scores.mean().round(3) , scores.std().round(3))

In [ ]:
pipe_lr.fit(X_train, y_train)
pred_lr = pipe_lr.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

r2 = r2_score(y_test, pred_lr)
print(r2)


In [ ]:
pipe_rf.fit(X_train, y_train)
pred_rf=pipe_rf.predict(X_test)

In [ ]:
r2 = r2_score(y_test, pred_rf)
print(r2)